2PC tackelt het probleem van transacties van een database dat gedistribueerd is over meerder servers of schijven. Problemen die komen kijken bij deze casus

- **Site failures**  
  Een site is operationeel of uitgevallen (fail/stop-gedrag).

- **Partial failure in the network**  
  Sommige sites zijn operationeel, terwijl andere sites uitgevallen zijn.

- **Total failure**  
  Alle sites zijn uitgevallen.

- **Communication failure**  
  Netwerkpartities of componenten die niet met elkaar kunnen communiceren.

#### Aannames

- **Undeliverable messages dropped**  
  Berichten die niet kunnen worden afgeleverd, worden verworpen.

- **Failure detection by timeout**  
  Fouten worden gedetecteerd via een timeout-mechanisme.

De twee fase commit protocol werkt als volgt, je hebt een COORDinator, dit is de source van de transactie. Denk aan een ING entiteit in Amsterdam dat een transactie wil uitvoeren in een database die in meerdere panden in het land gestationeerd zijn. Deze COORD vraag aan de PARTicipants, de andere fysiek aanwezige disks, sites, nodes of een andere database instance met betrekking tot dezelfde database, of ze deze transactie kunnen committen. Samengevat is de filosofie rond 2PC dan ook:

$$
    \text{Either all commit or all abort}
$$

D.w.z. samen thuis samen uit.

**Fase 1**

De COORD stuurt naar alle PART's: $\text{[prepare-T]}$, elke PART checkt dan of haar database volledig/valide is of zij kan committen en of zij resources kan locken. Mocht dit het geval zijn zal de PART het volgende terug sturen $\text{[ready T]}$, zo niet $\text{[abort T]}$. Waarbij $\text{T}$ symbolisch staat voor de transactie.

**Fase 2**

Na het voting proces zal de COORD, in het geval dat alle PART's $\text{[ready T]}$ hebben gestemd, een global commit uitvoeren $\text{GLOBAL COMMIT}$. Hierna zullen alle PART's en de COORD de transactie committen volgens de besproken methoden van een op zich zelf staande transactie. Zo niet zal de COORD een global abort uitvoeren $\text{GLOBAL ABORT}$, waarna alles PART's en de COORD een rollback zullen doen. Dit houdt in dat de transactie dus niet wordt uitgevoerd.

**DT-log**

*Fase 1*

Elke PART's (hier zit de COORD bij in) houden een DT-log bij, dit lijkt op de log die we kunnen uit recovery. Waneer de COORD aan fase 1 begint zal hij in zijn DT-log $\langle \text{prepare-T} \rangle$ zetten. 

 Wanneer de PART's de $\text{[prepare-T]}$ ontvangen, zullen zij $\langle \text{abort-T} \rangle$ loggen als zij abort stemmen en uiteraard $\langle \text{ready-T} \rangle$ loggen wanneer zij ready stemmen.

 *Fase 2*

 Wanneer de COORD alle stemmen verzameld en elke stem (inclusief die van de COORD) is positief zal de COORD $\langle \text{Commit-T} \rangle$ loggen. Is er een stem (of meer) die negatief zijn zal de COORD $\langle \text{Abort-T} \rangle$ loggen en alleen naar de positieve stemmen $\text{[Abort-T]}$ sturen.

**Wanneer gaat het fout?**

De reden van de DT-log is om een recovery proces mogelijk te maken wanneer er iets mis gaat tijdens het 2PC proces. 

Stel er gaat iets mis bij $A$, dit kan een PART zijn of de COORD zelf, $A$ ziet het volgende:

#### Wat ziet $A$ in haar log?

- Een $\langle \text{START } T \rangle$ record  
- van welke transactie $A$ deel uitmaakt
- weet wie de PART's zijn  

#### Wat ziet $A$ in haar distributed transaction (DT) log? Vijf mogelijkheden:

- Een $\langle \text{COMMIT } T \rangle$ record  
- Een $\langle \text{ABORT } T \rangle$ record  
- Een $\langle \text{PREPARE } T \rangle$ record  
- Een $\langle \text{READY } T \rangle$ record  
- Geen $\langle \text{READY } T \rangle$ record  

We gaan een aantal van de mogelijkheden af en zien wat we dan moeten doen.

### 1.

#### DT-log

- Een $\langle \text{COMMIT } T \rangle$ record, of  
- Een $\langle \text{ABORT } T \rangle$ record  

#### Actie:

- Blijkbaar is de beslissing al genomen  
- Handel volgens deze beslissing, wanneer commit $\rightarrow$ REDO, wanneer abort $\rightarrow$ UNDO

### 2.

#### DT-log

- Een $\langle \text{PREPARE } T \rangle$ record, geen decision record  

#### Actie:

**rol als COORD**
- $A$ heeft het protocol gestart, maar blijkbaar is er nog geen beslissing genomen  
- $A$ heeft de mogelijkheid om een abort af te dwingen  
- $A$ hervat het protocol met een abort-beslissing omdat $A$ niet weet hoe gestemd is

**rol als PART**
- $A$ ziet het $\langle \text{PREPARE } T \rangle$ in haar log, wat betekent dat $A$ yes heeft gestemd
- $A$ weet de globale beslissing niet en moet dus de COORD contacteren
- $A$ handelt naar wat de COORD zegt, commit of abort

### 3.

#### DT-log

- Geen $\langle \text{READY } T \rangle$ record, en geen decision record  

#### Actie:

**rol moet PART zijn**
- $A$ heeft nog geen $\langle \text{READY } T \rangle$ in haar log, dus $A$ heeft nog niet YES gestemd  
- $A$ weet dat er nog geen globale beslissing is genomen  
- $A$ heeft de mogelijkheid om een abort af te dwingen (door NO te stemmen)  
- $A$ hervat het protocol met een abort-beslissing omdat er iets mis is gegaan bij $A$

### 4.

#### DT-log

- Een $\langle \text{READY } T \rangle$ record, geen decision record  

#### Actie:

**rol moet PART zijn**
- $A$ ziet een $\langle \text{READY } T \rangle$ in haar log, wat betekent dat $A$ YES heeft gestemd  
- $A$ kent de globale beslissing niet en bevindt zich in een onzekere (blocking) toestand  
- $A$ probeert de COORD te contacteren om de beslissing te achterhalen  
- $A$ handelt naar de globale beslissing: commit of abort  

#### Cooperative Termination Protocol (CTP)

Het $\textbf{Cooperative Termination Protocol (CTP)}$ wordt gebruikt wanneer de COORD faalt
tijdens het Two-Phase Commit (2PC) protocol en de PART's in een onzekere toestand zitten.

#### Doel

Het doel van CTP is om alsnog een consistente globale beslissing (commit of abort) te bereiken
zonder de oorspronkelijke COORD.

#### Werking

- Kies een leider (leader) uit de overgebleven PART's  
- Deze leider fungeert als nieuwe COORD  
- De leider vraagt aan alle PART's hun status op (bijv. READY, COMMIT, ABORT, of geen stem)  

Vervolgens zijn er verschillende gevallen:

- Als een PART de $\textbf{oorspronkelijke beslissing}$ kent ($\text{COMMIT}$ of $\text{ABORT}$),  
  dan broadcast de leider deze beslissing naar alle PART's  

- Als een PART nog niet heeft gestemd (dus geen $\langle \text{READY } T \rangle$ heeft),  
  dan kan de leider veilig een $\textbf{ABORT}$ beslissen en broadcasten  

- Als alle PART's een $\langle \text{READY } T \rangle$ hebben (dus allemaal YES hebben gestemd),  
  maar niemand kent de uiteindelijke beslissing, dan ontstaat er een $\textbf{blocking situatie}$  

$$
\text{Iedereen wacht, maar niemand kan beslissen}
$$

#### Intuïtief inzicht

- $\textbf{READY}$ betekent: ``ik ben bereid te committen en wacht op de beslissing''  
- Zonder kennis van de globale beslissing kan geen enkele PART veilig kiezen tussen commit of abort  
- Daarom ontstaat er een $\textbf{deadlock-achtige situatie}$  

#### Blocking protocols

Een protocol is $\textbf{blocking}$ als processen moeten wachten op een gebeurtenis die mogelijk nooit plaatsvindt.

- $\textbf{2PC is een blocking protocol}$  
  Als de COORD faalt nadat PART's READY hebben gelogd, kunnen zij niet zelfstandig beslissen  

- Blocking kan leiden tot:
  - vasthouden van locks  
  - stilstaande systemen  
  - verminderde beschikbaarheid  

- Vaak is $\textbf{menselijke interventie}$ nodig (bijvoorbeeld van een DBA),  
  of speciale recovery/protocollen zoals CTP  

#### 3PC (Three-Phase Commit)

- $\textbf{3PC}$ is ontworpen om blocking te verminderen  
- Het introduceert een extra fase zodat processen nooit in een volledig onzekere toestand zitten  

Maar:

- Bij $\textbf{netwerkproblemen}$ (partitions) kan ook 3PC nog steeds blokkeren  
- 3PC vereist sterkere aannames over timing en communicatie  

#### Fundamenteel resultaat

Er bestaat een theoretisch resultaat dat stelt:

$$
\text{Een volledig non-blocking commit protocol is onmogelijk in een asynchroon systeem met failures}
$$

Dit betekent:

- In systemen zonder perfecte communicatie en timing  
- en met mogelijke crashes  

$$
\text{zal er altijd een scenario bestaan waarin processen moeten wachten (blocking)}
$$